# Notebook 03 -- Individual Laplacian Renormalization

This notebook runs Laplacian renormalization on three individual networks
(NetSci, Facebook, C. Elegans) and produces harmonic/conformal degree curves
and coarse-grained graph visualizations.

**Figures produced:**
- **Figure 12:** NetSci Collaboration -- harmonic degree curves and visualization
- **Figure 13:** Facebook -- harmonic degree curves and visualization
- **Figure 14:** C. Elegans -- harmonic degree curves and visualization

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from harmonic_morphisms import H_CF_curves, renorm_graph_harmonic, g_len
from harmonic_morphisms.core import renorm_graph_plot
from harmonic_morphisms.simplicial import import_network_data

DATA_DIR = Path("../data")

SMALL_SIZE = 10; MEDIUM_SIZE = 12; BIGGER_SIZE = 13
plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=SMALL_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

In [ ]:
# Network configurations:
# (filename, display_name, threshold, visualization_time, figure_number)
networks = [
    ("out.dimacs10-netscience", "NetSci Collaboration", pow(10, -3), 1, 12),
    ("out.ego-facebook", "Facebook", pow(10, -3), 1.95, 13),
    ("out.dimacs10-celegans_metabolic", "C. Elegans", pow(10, -3), 1, 14),
]

for filename, display_name, tresh, viz_time, fig_num in networks:
    print(f"\n{'='*60}")
    print(f"Processing {display_name} (Figure {fig_num})")
    print(f"{'='*60}")

    # --- Load network ---
    f = open(DATA_DIR / "networks" / filename)
    Ag = import_network_data(f)
    print(f"{display_name}: {Ag.number_of_nodes()} nodes, {Ag.number_of_edges()} edges")

    # --- Compute Laplacian and harmonic/conformal degree curves ---
    L0 = nx.laplacian_matrix(Ag, nodelist=Ag.nodes()).todense()
    GRAPHS_l, DEG_H_L, M_DEG_H_l, STD_H_l, AV_H_l, STD_V_H_l, NOT_H_l, \
        DEG_CF_l, M_DEG_CF_l, STD_CF_l, AV_CF_l, STD_V_CF_l, NOT_CF_l, \
        gV_l, t_h_l = H_CF_curves(Ag, L0, 100, tresh)
    compression = 1 - np.array(g_len(GRAPHS_l)) / len(Ag.nodes())
    print(f"Collapse time: {t_h_l[-1]:.2f}, steps: {len(GRAPHS_l)}")

    # --- Figure: Harmonic/conformal degree curves vs compression ---
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(compression, M_DEG_H_l, "ob--", markersize=4, label=r"$\hat{H}$")
    ax.plot(compression, M_DEG_CF_l, "og--", markersize=4, label=r"$\hat{CF}$")
    ax.plot(compression, STD_H_l, "ok--", markersize=4, label=r"$H_{Dev}$")
    ax.set_xlabel("Compression")
    ax.set_ylabel("Measure")
    ax.set_ylim([-0.1, 1.2])
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title(f"Figure {fig_num}: Laplacian RG on {display_name}")
    plt.tight_layout()
    plt.savefig(f"figure_{fig_num}_laplacian_{display_name.replace(' ', '_').lower()}.pdf",
                bbox_inches="tight")
    plt.show()

    # --- Visualization: original clustering + coarse-grained graph ---
    G, deg_h, M_deg_h, std_h, av_h, std_v_h, Not_h, \
        deg_cf, M_deg_cf, std_cf, av_cf, std_v_cf, Not_cf, Gv = \
        renorm_graph_harmonic(Ag, viz_time, L0, tresh)

    G_plot, colors_d, sing_col = renorm_graph_plot(Ag, viz_time, L0, tresh)
    colors = [colors_d[n] for n in Ag.nodes()]

    layout = nx.spring_layout(Ag, iterations=100, seed=42)
    lay2 = nx.spring_layout(G_plot, iterations=100, seed=42)

    f2, ax2 = plt.subplots(1, 2, figsize=(8.3, 4))
    ax2 = ax2.flatten()

    nodes = nx.draw_networkx_nodes(Ag, ax=ax2[0], pos=layout, node_color=colors, node_size=14)
    nodes.set_edgecolor("white")
    nx.draw_networkx_edges(Ag, ax=ax2[0], pos=layout, width=0.8)
    ax2[0].collections[0].set_linewidth(0.7)
    ax2[0].collections[0].set_edgecolor("#FFFFFF")
    ax2[0].set_title("Clustering", fontsize=10)

    nodes_2 = nx.draw_networkx_nodes(G_plot, ax=ax2[1], pos=lay2, node_color=sing_col, node_size=60)
    nodes_2.set_edgecolor("white")
    nx.draw_networkx_edges(G_plot, ax=ax2[1], pos=lay2, width=1.2)
    ax2[1].collections[0].set_linewidth(0.6)
    ax2[1].collections[0].set_edgecolor("#FFFFFF")
    comp = 1 - len(G_plot.nodes) / len(Ag.nodes)
    ax2[1].set_title(
        f"Laplacian, time {viz_time}, compression {comp:.2f}",
        fontsize=10,
    )

    f2.suptitle(
        f"Figure {fig_num}: {display_name} -- Laplacian Renormalization at t={viz_time}",
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(f"figure_{fig_num}_viz_{display_name.replace(' ', '_').lower()}.pdf",
                bbox_inches="tight")
    plt.show()

    # --- Metrics table ---
    Values = [deg_h, M_deg_h, std_h, deg_cf, M_deg_cf, std_cf]
    Labels = ["H", "Mod. H", "Std. H", "CF", "Mod. CF", "Std. CF"]
    df = pd.DataFrame([Values], columns=Labels, index=[display_name])
    print(f"\nMetrics at t={viz_time}:")
    print(df.to_string())